# Stroke Prediction

**Objective**: Build machine learning models to predict stroke likelihood.

**Challenge**: The dataset has a severe class imbalance (very few stroke cases compared to non-stroke cases). We will handle this using SMOTE and class weights to see which works better.


## 1. Import Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, precision_score, recall_score
)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Create a folder for saving plots
if not os.path.exists('outputs'):
    os.makedirs('outputs')

print("Libraries imported.")


## 2. Data Loading & EDA


In [ ]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", list(df.columns))

# Show first few rows
print("\nFirst 5 Rows:")
print(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

# Note on BMI
print("\nNote: 'bmi' column contains 'N/A' strings instead of empty values.")
na_bmi_count = (df['bmi'] == 'N/A').sum()
print(f"Count of 'N/A' in bmi: {na_bmi_count}")

# Class distribution
stroke_counts = df['stroke'].value_counts()
stroke_pct = df['stroke'].value_counts(normalize=True) * 100

print("\nTarget Variable (stroke) Distribution:")
print(f"No Stroke (0): {stroke_counts[0]} ({stroke_pct[0]:.2f}%)")
print(f"Stroke (1): {stroke_counts[1]} ({stroke_pct[1]:.2f}%)")
print("\nSince the data is heavily imbalanced, accuracy is not a good metric here.")


In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].bar(['No Stroke', 'Stroke'], stroke_counts.values, color=['skyblue', 'salmon'])
axes[0].set_title('Class Count')
axes[0].set_ylabel('Number of Patients')

axes[1].pie(stroke_counts.values, labels=['No Stroke', 'Stroke'], colors=['skyblue', 'salmon'], autopct='%1.1f%%')
axes[1].set_title('Class Percentage')

plt.tight_layout()
plt.savefig('outputs/class_distribution.png')
plt.show()


## 3. Data Preprocessing


In [ ]:
print("\nStarting preprocessing...")

# Drop the id column because it doesn't help with predictions
df = df.drop('id', axis=1)

# Drop gender='Other' since there is only 1 record
other_count = (df['gender'] == 'Other').sum()
df = df[df['gender'] != 'Other'].reset_index(drop=True)

# Convert bmi to numeric and fill missing values with the median
df['bmi'] = pd.to_numeric(df['bmi'], errors='coerce')
bmi_median = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(bmi_median)
print(f"Filled missing BMI values with median: {bmi_median:.1f}")

# Categorical columns to encode
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

# One-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# convert boolean to int
bool_columns = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_columns] = df_encoded[bool_columns].astype(int)

# Split features (X) and target (y)
X = df_encoded.drop('stroke', axis=1)
y = df_encoded['stroke']

# Train-test split (80% train, 20% test)
# stratify=y ensures the same stroke percentage in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# Scale numeric features
numeric_features = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print("Preprocessing complete.")


## 4. Handling Imbalance

We'll use two methods:
1. **SMOTE**: Generates synthetic data for the minority class in the training set.
2. **Class Weights**: Gives higher penalty to minority class errors during training.


In [ ]:
# Method 1: SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"SMOTE Training data shape: {X_train_smote.shape}")
print(f"SMOTE target counts:\n{y_train_smote.value_counts()}")

# For Method 2, we just use the class_weight parameter when building models.


## 5. Training Models


In [ ]:
# Calculate weight for XGBoost
neg_class = (y_train == 0).sum()
pos_class = (y_train == 1).sum()
scale_weight = neg_class / pos_class

models = {}

# 1. Models trained on SMOTE data
print("\nTraining SMOTE models...")
models['LR_SMOTE'] = LogisticRegression(max_iter=1000, random_state=42)
models['RF_SMOTE'] = RandomForestClassifier(n_estimators=100, random_state=42)
models['XGB_SMOTE'] = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')

for name in ['LR_SMOTE', 'RF_SMOTE', 'XGB_SMOTE']:
    models[name].fit(X_train_smote, y_train_smote)
    print(f"{name} trained.")

# 2. Models trained with class weights (on original scaled data)
print("\nTraining ClassWeight models...")
models['LR_ClassWeight'] = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
models['RF_ClassWeight'] = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
models['XGB_ClassWeight'] = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', scale_pos_weight=scale_weight)

for name in ['LR_ClassWeight', 'RF_ClassWeight', 'XGB_ClassWeight']:
    models[name].fit(X_train_scaled, y_train)
    print(f"{name} trained.")

print("All models trained successfully.")


## 6. Model Evaluation

We evaluate on the test set, focusing on Recall and F1-Score because we want to catch as many stroke cases as possible (Recall).


In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        'Model': name,
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4)
    })

# Convert results to dataframe and sort by F1
results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
results_df.to_csv('outputs/model_results.csv', index=False)

print("\nModel Results:")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f"\nThe best model based on F1-Score is: {best_model_name}")


## 7. Visualizations


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(name)
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('outputs/confusion_matrices.png')
plt.show()


## 8. Test With User Input

Below is a function to test the best model with manual input data.


In [ ]:
def predict_stroke_interactive():
    print("\n--- Stroke Prediction Test ---")
    print("Please enter the patient details:")
    
    try:
        age = float(input("Age: "))
        avg_glucose = float(input("Average glucose level (e.g. 100): "))
        bmi_input = float(input("BMI (e.g. 25.5): "))
        
        gender = input("Gender (Male/Female): ").strip().capitalize()
        hypertension = int(input("Hypertension (0=No, 1=Yes): "))
        heart_disease = int(input("Heart Disease (0=No, 1=Yes): "))
        ever_married = input("Ever Married? (Yes/No): ").strip().capitalize()
        work_type = input("Work Type (Private/Self-employed/Govt_job/children/Never_worked): ").strip()
        residence = input("Residence Type (Urban/Rural): ").strip().capitalize()
        smoking = input("Smoking Status (never smoked/formerly smoked/smokes/Unknown): ").strip()

        # Create a zero dataframe matching the training columns
        input_data = pd.DataFrame(0, index=[0], columns=X.columns)
        
        # Set numeric values
        input_data['age'] = age
        input_data['avg_glucose_level'] = avg_glucose
        input_data['bmi'] = bmi_input
        input_data['hypertension'] = hypertension
        input_data['heart_disease'] = heart_disease
        
        # Set encoded categorical values
        if f'gender_{gender}' in input_data.columns:
            input_data[f'gender_{gender}'] = 1
        if f'ever_married_{ever_married}' in input_data.columns:
            input_data[f'ever_married_{ever_married}'] = 1
        if f'work_type_{work_type}' in input_data.columns:
            input_data[f'work_type_{work_type}'] = 1
        if f'Residence_type_{residence}' in input_data.columns:
            input_data[f'Residence_type_{residence}'] = 1
        if f'smoking_status_{smoking}' in input_data.columns:
            input_data[f'smoking_status_{smoking}'] = 1
            
        # Scale the numeric inputs using our fitted scaler
        input_data[numeric_features] = scaler.transform(input_data[numeric_features])
        
        # Make prediction using the best model
        best_model = models[best_model_name]
        pred = best_model.predict(input_data)[0]
        prob = best_model.predict_proba(input_data)[0][1]
        
        print("\n--- Result ---")
        if pred == 1:
            print(f"Prediction: STROKE RISK DETECTED (Probability: {prob*100:.1f}%)")
        else:
            print(f"Prediction: LOW RISK (Probability: {prob*100:.1f}%)")
            
    except Exception as e:
        print("Error with input:", e)


In [ ]:
# Run the interactive test below
predict_stroke_interactive()
